In [1]:
import re
import emoji
import joblib

import numpy as np
from pythainlp.tokenize import word_tokenize
from scipy.sparse import hstack

from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

In [2]:
def clean_text(text):
    text = str(text)
    text = re.sub(r'&#\d+;', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' <URL> ', text)
    text = re.sub(r'@\S+', ' <USER> ', text)
    text = emoji.replace_emoji(text, replace=' <EMOJI> ')
    text = re.sub(r'[^ก-๙a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def thai_tokenizer(text):
    return word_tokenize(text, engine="newmm")

In [3]:
import pandas as pd

df = pd.read_json("../../dataset/train_sentiment.json")

# fix JSON orientation if needed
if "text" not in df.columns:
    df = df.transpose().reset_index(drop=True)

df.head()

,text,sentiment
0,มาตามนัด 🚮🗑️ . . #เขตพญาไท #กรุงเทพมหานคร @สาย...,positive
1,&#9888; แยกราชเทวีจะรื้อน้ำพุกี่โมง? &#128591;...,negative
2,"&#128308;สด!""ไอซ์ รักชนก"" แถลงข่าวการประมูลคลื...",negative
3,&#127808;#เขตหนองจอก >>ติดตามการดำเนินงานโครงก...,neutral
4,ไทวัสดุ ส่งช่างมือ 1 วีฟิกซ์ ร่วมฟื้นฟู ปรับภู...,positive


In [4]:
df["clean_text"] = df["text"].apply(clean_text)

X = df["clean_text"]
y = df["sentiment"]

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# word-level TF-IDF
tfidf_word = TfidfVectorizer(
    tokenizer=thai_tokenizer,
    ngram_range=(1, 2),
    min_df=3,
    max_features=50000
)

X_word_train = tfidf_word.fit_transform(X_train)
X_word_test = tfidf_word.transform(X_test)

C:\Users\watta\anaconda3\envs\thai-sentiment\lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [7]:
# character-level TF-IDF
tfidf_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3,
    max_features=30000
)

X_char_train = tfidf_char.fit_transform(X_train)
X_char_test = tfidf_char.transform(X_test)

In [8]:
from scipy.sparse import hstack

X_train_final = hstack([X_word_train, X_char_train])
X_test_final = hstack([X_word_test, X_char_test])

In [9]:
svm_model = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train_final, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,verbose,0
,random_state,42


In [10]:
y_pred = svm_model.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7827777777777778
              precision    recall  f1-score   support

    negative       0.84      0.84      0.84       600
     neutral       0.71      0.69      0.70       600
    positive       0.80      0.82      0.81       600

    accuracy                           0.78      1800
   macro avg       0.78      0.78      0.78      1800
weighted avg       0.78      0.78      0.78      1800

[[505  77  18]
 [ 80 411 109]
 [ 17  90 493]]


In [11]:
import joblib

# save Linear SVM model
joblib.dump(svm_model, "linear_svm_model.pkl")

# save TF-IDF vectorizers
joblib.dump(tfidf_word, "tfidf_word_svm.pkl")
joblib.dump(tfidf_char, "tfidf_char_svm.pkl")

# save label encoder
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']